# Future Enhancements: Advanced Deadlock Simulator Features

This notebook explores the conceptual design, algorithmic extensions, and technical implementation strategies for the three core future enhancements proposed for the Deadlock Detection and Recovery Simulator.

## Enhancements Covered:
1. **Interactive Resource Allocation Graph (RAG)**
2. **Multi-Instance Resources & Wait-For Graphs**
3. **Rollback-Based Recovery Simulation**

---
## 1. Graphical, Interactive RAG with Drag-and-Drop

**Objective:** Transform the static, view-only RAG into a dynamic playground where users can visually manipulate the system state by dragging and dropping edges between processes and resources.

### Technical Approach
* **Frontend:** Utilize a library like **Cytoscape.js** or **D3.js** to render nodes (Processes as circles, Resources as squares).
* **Interactions:** 
  * Dragging a line from a Process to a Resource creates a `Request Edge`.
  * Dragging a line from a Resource to a Process creates an `Assignment Edge`.
* **Backend Sync:** Each drag-and-drop action triggers an API call to update the backend Adjacency Matrix, immediately followed by a cycle-detection check to visualize deadlocks in real-time (e.g., highlighting cycle edges in red).

### Example: Backend Mockup using NetworkX
Below is a conceptual Python mockup demonstrating how the backend would handle dynamic edge additions and immediately check for cycles.

In [ ]:
import networkx as nx

class InteractiveRAG:
    def __init__(self):
        self.graph = nx.DiGraph()
        
    def add_process(self, p_id):
        self.graph.add_node(p_id, type='process', color='skyblue')
        
    def add_resource(self, r_id):
        self.graph.add_node(r_id, type='resource', color='lightgreen')
        
    def add_request_edge(self, p_id, r_id):
        # Process requests Resource
        self.graph.add_edge(p_id, r_id, edge_type='request')
        print(f"Added Request: {p_id} -> {r_id}")
        return self.check_deadlock()
        
    def add_assignment_edge(self, r_id, p_id):
        # Resource assigned to Process
        self.graph.add_edge(r_id, p_id, edge_type='assignment')
        print(f"Added Assignment: {r_id} -> {p_id}")
        return self.check_deadlock()

    def check_deadlock(self):
        try:
            # Detect cycle in directed graph
            cycle = nx.find_cycle(self.graph, orientation='original')
            print(f"⚠️ DEADLOCK DETECTED! Cycle: {cycle}")
            return True, cycle
        except nx.NetworkXNoCycle:
            print("✅ System is Safe. No cycles detected.")
            return False, []

# Simulate User Drag-and-Drop
rag = InteractiveRAG()
rag.add_process("P1")
rag.add_process("P2")
rag.add_resource("R1")
rag.add_resource("R2")

# Normal assignments and requests
rag.add_assignment_edge("R1", "P1")
rag.add_request_edge("P1", "R2")
rag.add_assignment_edge("R2", "P2")

# User drags P2 to R1 creating a cycle
print("\n--- User Drag & Drop: P2 requests R1 ---")
is_deadlock, cycle = rag.add_request_edge("P2", "R1")

---
## 2. Multi-Instance Resources with Full Wait-For Graph (WFG) Detection

**Objective:** Upgrade from a single-instance constraint (where a simple cycle equals deadlock) to a multi-instance environment where a cycle is *necessary but not sufficient* for deadlock. 

### Technical Approach
* **Algorithm Transition:** Shift from standard DFS cycle detection to **Graph Reduction algorithms** or the **Banker's Algorithm** (for deadlock avoidance/detection).
* **Wait-For Graph (WFG):** If all resources have only a single instance, we can collapse the RAG into a Wait-For Graph (removing resource nodes and connecting processes directly). For multi-instance, we maintain the full RAG but apply reduction rules:
  1. Find a process whose requests can be satisfied by currently available resource instances.
  2. Simulate process completion and release its resources.
  3. Repeat until either all processes complete (Safe) or no further reductions can be made (Deadlock).

### Example: Multi-Instance Graph Reduction Simulator

In [ ]:
import numpy as np

class MultiInstanceDeadlockDetector:
    def __init__(self, num_processes, num_resources):
        self.P = num_processes
        self.R = num_resources
        self.available = np.zeros(self.R, dtype=int)
        self.allocation = np.zeros((self.P, self.R), dtype=int)
        self.request = np.zeros((self.P, self.R), dtype=int)

    def set_state(self, available, allocation, request):
        self.available = np.array(available)
        self.allocation = np.array(allocation)
        self.request = np.array(request)

    def detect_deadlock(self):
        work = self.available.copy()
        finish = [False] * self.P
        deadlocked_processes = []

        # Optimization: Processes with no allocation and no requests are trivially finished
        for i in range(self.P):
            if sum(self.allocation[i]) == 0 and sum(self.request[i]) == 0:
                finish[i] = True

        while True:
            progress_made = False
            for i in range(self.P):
                if not finish[i] and all(self.request[i] <= work):
                    # Process can finish
                    work += self.allocation[i]
                    finish[i] = True
                    progress_made = True
                    print(f"Process P{i} can complete. New Available: {work}")
            
            if not progress_made:
                break

        if all(finish):
            print("\n✅ System is SAFE. No Deadlock.")
        else:
            deadlocked_processes = [f"P{i}" for i, f in enumerate(finish) if not f]
            print(f"\n⚠️ DEADLOCK DETECTED! Deadlocked Processes: {', '.join(deadlocked_processes)}")

# Scenario
detector = MultiInstanceDeadlockDetector(3, 3)
# R0: 3, R1: 2, R2: 2 total instances
avail = [0, 0, 0] 
alloc = [
    [2, 0, 0], # P0
    [1, 2, 0], # P1
    [0, 0, 2]  # P2
]
req = [
    [0, 1, 0], # P0 requests R1
    [0, 0, 1], # P1 requests R2
    [1, 0, 0]  # P2 requests R0
]
print("--- Evaluating Multi-Instance State ---")
detector.set_state(avail, alloc, req)
detector.detect_deadlock()

---
## 3. Simulate Rollback-Based Recovery

**Objective:** Currently, recovery involves terminating processes completely. A more advanced and realistic OS approach is **Checkpointing and Rollback**, where a process is rewound to a previous safe state rather than being killed outright.

### Technical Approach
* **Checkpointing Mechanism:** Periodically save the state of a process (allocated resources, execution progress). 
* **Rollback Logic:** When a deadlock is detected, select a "victim" process based on cost. Instead of aborting it, revert its state to the most recent checkpoint that breaks the deadlock cycle.
* **Cost Analysis:** Implement algorithms to evaluate the "Cost of Rollback" (e.g., number of rolled-back operations, priority of process) versus "Cost of Termination".

### Example: Checkpoint & Rollback Simulation

In [ ]:
import copy

class ProcessSimulator:
    def __init__(self, p_id):
        self.p_id = p_id
        self.resources_held = []
        self.checkpoints = [] # Stack of safe states

    def save_checkpoint(self):
        # Save a deep copy of current resources
        state = copy.deepcopy(self.resources_held)
        self.checkpoints.append(state)
        print(f"[{self.p_id}] Checkpoint saved. State: {state}")

    def acquire_resource(self, r_id):
        self.resources_held.append(r_id)
        print(f"[{self.p_id}] Acquired {r_id}. Current holding: {self.resources_held}")

    def rollback(self):
        if self.checkpoints:
            # Revert to the last safe state
            last_state = self.checkpoints.pop()
            released = [r for r in self.resources_held if r not in last_state]
            self.resources_held = last_state
            print(f"[{self.p_id}] ⚠️ ROLLED BACK! Released: {released}. Reverted to: {self.resources_held}")
            return released
        else:
            print(f"[{self.p_id}] ❌ No checkpoints available. Terminating process.")
            released = self.resources_held
            self.resources_held = []
            return released

# Simulate execution
print("--- Simulating Process Execution & Checkpointing ---")
p1 = ProcessSimulator("P1")

p1.acquire_resource("R1")
p1.save_checkpoint() # State: ['R1']

p1.acquire_resource("R2")
p1.save_checkpoint() # State: ['R1', 'R2']

p1.acquire_resource("R3") # Deadlock triggers here!

print("\n--- Deadlock Detected! Initiating Rollback for P1 ---")
released_resources = p1.rollback()
print(f"Resources returned to available pool: {released_resources}")

# If still deadlocked, rollback further
print("\n--- Cycle still exists! Rolling back further ---")
released_resources = p1.rollback()
print(f"Resources returned to available pool: {released_resources}")

---
## Conclusion

These future enhancements will transform the simulator from a basic academic tool into an advanced, comprehensive Operating System laboratory platform.
* The **Interactive RAG** will dramatically improve pedagogical value, allowing students to "feel" how deadlocks occur.
* Supporting **Multi-Instance resources** ensures the simulator aligns fully with real-world OS resource managers.
* **Rollback Recovery** introduces advanced OS concepts like state preservation, checkpointing, and cost-based victim selection.